# Template
## PromptTemplate and from_template()

In [1]:
from langchain_core.prompts import PromptTemplate

# creating a prompt template with input variables
prompt_template = PromptTemplate.from_template(
    "Tell me a {adjective} joke about {content}."
)

In [6]:
prompt_template.input_variables

['adjective', 'content']

In [8]:
prompt = prompt_template.format(adjective="funny", content="chickens")
print(type(prompt))
print(prompt)

<class 'str'>
Tell me a funny joke about chickens.


In [2]:
prompt_template.format(adjective="silly", content="AI")

'Tell me a silly joke about AI.'

In [3]:
# we can use StringPromptValue and convert it to List of ChatMessages
prompt = prompt_template.format_prompt(adjective="funny", content="chickens").to_messages()
print(type(prompt))
print(prompt)

<class 'list'>
[HumanMessage(content='Tell me a funny joke about chickens.', additional_kwargs={}, response_metadata={})]


# ChatPromptTemplate and from_template()

In [4]:
from langchain_core.prompts import ChatPromptTemplate
chat_template = ChatPromptTemplate.from_template(
    "What is {topic}?")

In [17]:
chat_template.input_variables

['topic']

In [19]:
prompt = chat_template.format(topic="machine learning")
print(type(prompt))
print(prompt)

<class 'str'>
Human: What is machine learning?


In [21]:
prompt = chat_template.format_messages(topic="machine learning")
print(type(prompt))
print(prompt)

<class 'list'>
[HumanMessage(content='What is machine learning?', additional_kwargs={}, response_metadata={})]


In [23]:
prompt = chat_template.format_prompt(topic="machine learning")
print(type(prompt))
print(prompt)

<class 'langchain_core.prompt_values.ChatPromptValue'>
messages=[HumanMessage(content='What is machine learning?', additional_kwargs={}, response_metadata={})]


# ChatPromptTemplate and from_messages()

In [26]:
from langchain_core.prompts import ChatPromptTemplate
chat_template = ChatPromptTemplate.from_messages(
    [
        ("system","you are a helpful AI Bot. Your name is {name}."),
        ("human","Hello, how are you doing today?"),
        ("ai","I'm doing well, thanks!"),
        ("human","{user_input}"),        
    ]
    
)

In [28]:
chat_template.input_variables

['name', 'user_input']

In [30]:
chat_template.format(name="sundram", user_input="what is your name?")

"System: you are a helpful AI Bot. Your name is sundram.\nHuman: Hello, how are you doing today?\nAI: I'm doing well, thanks!\nHuman: what is your name?"

In [32]:
chat_template.format_prompt(name="sundram", user_input="what is your name?")

ChatPromptValue(messages=[SystemMessage(content='you are a helpful AI Bot. Your name is sundram.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello, how are you doing today?', additional_kwargs={}, response_metadata={}), AIMessage(content="I'm doing well, thanks!", additional_kwargs={}, response_metadata={}), HumanMessage(content='what is your name?', additional_kwargs={}, response_metadata={})])

### Designing ChatPromptTemplate using SystemMessagePromptTemplate, HumanMessagePromptTemplate and AIMessagePromptTemplate

In [36]:
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate,AIMessagePromptTemplate 

In [42]:
# system templete
system_prompt_template = SystemMessagePromptTemplate.from_template("You are a helpful AI Bot. Your name is {name}.")

# Human Template
human_prompt_template = HumanMessagePromptTemplate.from_template("Hello, how are you doing today?")

# AI Template
ai_prompt_template = AIMessagePromptTemplate.from_template("I'm doing well, thanks!")

# Human template
human_prompt_template_input = HumanMessagePromptTemplate.from_template("{user_input}")

# Compile a chat prompt

chat_template = ChatPromptTemplate.from_messages(
    [system_prompt_template, human_prompt_template,ai_prompt_template,human_prompt_template_input])

chat_template.format_prompt(name="Sundram", user_input="what is your name?").to_messages()

[SystemMessage(content='You are a helpful AI Bot. Your name is Sundram.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello, how are you doing today?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="I'm doing well, thanks!", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is your name?', additional_kwargs={}, response_metadata={})]

# LCEL - LangChain Expression Language

In [45]:
prompt = chat_template.invoke({"name":"Sundram", "user_input":"what is your name?"})
print(type(prompt))
print("****************")
print(prompt)


<class 'langchain_core.prompt_values.ChatPromptValue'>
****************
messages=[SystemMessage(content='You are a helpful AI Bot. Your name is Sundram.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello, how are you doing today?', additional_kwargs={}, response_metadata={}), AIMessage(content="I'm doing well, thanks!", additional_kwargs={}, response_metadata={}), HumanMessage(content='what is your name?', additional_kwargs={}, response_metadata={})]


In [47]:
prompt.to_string()

"System: You are a helpful AI Bot. Your name is Sundram.\nHuman: Hello, how are you doing today?\nAI: I'm doing well, thanks!\nHuman: what is your name?"

In [49]:
prompt.to_messages()

[SystemMessage(content='You are a helpful AI Bot. Your name is Sundram.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello, how are you doing today?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="I'm doing well, thanks!", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is your name?', additional_kwargs={}, response_metadata={})]

# Output Parsing

## CSV Parser (comma-seperated items)

In [57]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
csv_output_parser = CommaSeparatedListOutputParser()

In [59]:
#get_format_instructions()
csv_output_parser.get_format_instructions()

'Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'

In [61]:
examples = "Python, DA, DS, DL, ML, SQL, TABLEAU, POWERBI"
type(examples)

str

In [63]:
# .parser
csv_output_parser.parse(examples)

['Python', 'DA', 'DS', 'DL', 'ML', 'SQL', 'TABLEAU', 'POWERBI']

# examples

## PART 1 - Model (ChatLLM Model)

In [75]:
from langchain_openai import ChatOpenAI
f = open("keys/.openai_api_key.txt")
OPENAI_API_KEY = f.read()
chat_model = ChatOpenAI(api_key = OPENAI_API_KEY, model="gpt-4o-mini")

## PART 2 - Template (ChatPromptTemplate)

In [81]:
from langchain_core.prompts import ChatPromptTemplate
prompt_template = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful AI Chef Assistant.
                Given a dish name by user, you can provide the ingredients to prepare the dish.
                Output Format Instructions:
                {output_format_instructions}"""),
    ("human","Give me the ingredients for cooking {dish_name}."),
])
prompt_template

ChatPromptTemplate(input_variables=['dish_name', 'output_format_instructions'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['output_format_instructions'], input_types={}, partial_variables={}, template='You are a helpful AI Chef Assistant.\n                Given a dish name by user, you can provide the ingredients to prepare the dish.\n                Output Format Instructions:\n                {output_format_instructions}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['dish_name'], input_types={}, partial_variables={}, template='Give me the ingredients for cooking {dish_name}.'), additional_kwargs={})])

# Part 3 - Output Parser

In [84]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
csv_output_parser = CommaSeparatedListOutputParser()

# Part 4 - Chain (| - pipe sysmbol means chain or connector)

In [87]:
chain = prompt_template | chat_model | csv_output_parser

In [89]:
user_input = {"dish_name":"chicken biryani",  "output_format_instructions":csv_output_parser.get_format_instructions()}
response = chain.invoke(user_input)
print(response)

['rice', 'chicken', 'onions', 'garlic', 'ginger', 'tomatoes', 'yogurt', 'biryani spices (cumin', 'coriander', 'cardamom', 'cloves', 'cinnamon)', 'saffron', 'green chilies', 'cilantro', 'mint', 'lemon', 'cooking oil or ghee', 'salt', 'water']


In [91]:
user_input = {"dish_name":"paneer paratha",  "output_format_instructions":csv_output_parser.get_format_instructions()}
response = chain.invoke(user_input)
print(response)

['whole wheat flour', 'paneer', 'green chilies', 'ginger', 'coriander leaves', 'salt', 'oil', 'water', 'ghee']


# JSON Parser

In [94]:
from langchain_core.output_parsers import JsonOutputParser
json_output_parser = JsonOutputParser()

In [96]:
json_output_parser.get_format_instructions()

'Return a JSON object.'

In [100]:
from langchain_core.pydantic_v1 import BaseModel, Field

class QueryResponse(BaseModel):
    item_i: str = Field(description="Brief description and in how much quantity ith item should be used")

json_output_parser = JsonOutputParser(pydantic_object=QueryResponse)
print(json_output_parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"item_i": {"title": "Item I", "description": "Brief description and in how much quantity ith item should be used", "type": "string"}}, "required": ["item_i"]}
```


In [110]:
# just changing output parser from csv to json
json_chain = chain = prompt_template | chat_model | json_output_parser

In [112]:
user_input = {"dish_name":"paneer biryani",  "output_format_instructions":json_output_parser.get_format_instructions()}
response = chain.invoke(user_input)
print(response)

{'item_i': 'Basmati rice - 2 cups, Paneer (cottage cheese) - 250 grams, Onion - 2 medium (sliced), Tomato - 1 medium (chopped), Ginger-garlic paste - 1 tablespoon, Biryani masala - 2 tablespoons, Turmeric powder - 1/2 teaspoon, Red chili powder - 1 teaspoon, Green chilies - 2 (slit), Garam masala - 1 teaspoon, Fresh coriander leaves - 1/4 cup (chopped), Fresh mint leaves - 1/4 cup (chopped), Cooking oil or ghee - 3 tablespoons, Salt - to taste, Water - 4 cups'}


In [108]:
response

{'item_i': 'Basmati rice (1.5 cups), Paneer (250 grams), Onion (2 large, thinly sliced), Tomato (1 large, chopped), Ginger-garlic paste (2 tablespoons), Biryani masala (2 tablespoons), Yogurt (1/2 cup), Mint leaves (1/2 cup, chopped), Coriander leaves (1/2 cup, chopped), Green chilies (2, slit), Whole spices (bay leaf, cinnamon, cloves, cardamom), Ghee or oil (3 tablespoons), Salt (to taste), Water (3 cups)'}

# DateTime Parser

In [131]:
from langchain.output_parsers import DatetimeOutputParser
date_output_parser = DatetimeOutputParser()

In [135]:
date_output_parser.get_format_instructions()

"Write a datetime string that matches the following pattern: '%Y-%m-%dT%H:%M:%S.%fZ'.\n\nExamples: 2023-07-04T14:30:00.000000Z, 1999-12-31T23:59:59.999999Z, 2025-01-01T00:00:00.000000Z\n\nReturn ONLY this string, no other words!"

In [129]:
from langchain_core.prompts import PromptTemplate
template = """Answer the user question:
    Question:
    {question}

    Output Format Instructions:
    {output_format_instructions}"""

prompt_template = PromptTemplate.from_template(template)

In [133]:
chain = prompt_template | chat_model | date_output_parser

In [137]:
input = {"question": "What is Indian Independence Day?", "output_format_instructions": date_output_parser.get_format_instructions()}
response = chain.invoke(input)
response

datetime.datetime(1947, 8, 15, 0, 0)

# Pydantic Parser

In [164]:
!pip install pydantic==2.11.9

     ---------------------------------------- 0.0/68.4 kB ? eta -:--:--
     ---------------------------------------- 68.4/68.4 kB 1.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/444.9 kB ? eta -:--:--
   --------------------------------------  440.3/444.9 kB 13.9 MB/s eta 0:00:01
   ---------------------------------------- 444.9/444.9 kB 9.2 MB/s eta 0:00:00
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.11.7
    Uninstalling pydantic-2.11.7:
      Successfully uninstalled pydantic-2.11.7


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crewai 0.80.0 requires regex>=2024.9.11, but you have regex 2023.12.25 which is incompatible.
deepeval 1.4.7 requires grpcio~=1.63.0, but you have grpcio 1.70.0 which is incompatible.
deepeval 1.4.7 requires opentelemetry-api~=1.24.0, but you have opentelemetry-api 1.32.1 which is incompatible.
deepeval 1.4.7 requires opentelemetry-exporter-otlp-proto-grpc~=1.24.0, but you have opentelemetry-exporter-otlp-proto-grpc 1.32.1 which is incompatible.
deepeval 1.4.7 requires opentelemetry-sdk~=1.24.0, but you have opentelemetry-sdk 1.32.1 which is incompatible.
deepeval 1.4.7 requires tenacity~=8.4.1, but you have tenacity 8.5.0 which is incompatible.
embedchain 0.1.128 requires chromadb<0.6.0,>=0.5.10, but you have chromadb 0.5.0 which is incompatible.
embedchain 0.1.128 requires langchain-community<0.4.0,>=0.3.1, but 

In [165]:
"""
from langchain.output_parsers import PydanticOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field

class Song(BaseModel):
    name: str = Field(description="Name of the Song")
    geners: list = Field(description="List of the Geners")

parser = PydanticOutputParser(pydantic_object=Song)
parser.get_format_instructions()"""

'\nfrom langchain.output_parsers import PydanticOutputParser\nfrom langchain_core.pydantic_v1 import BaseModel, Field\n\nclass Song(BaseModel):\n    name: str = Field(description="Name of the Song")\n    geners: list = Field(description="List of the Geners")\n\nparser = PydanticOutputParser(pydantic_object=Song)\nparser.get_format_instructions()'

In [186]:
from pydantic import BaseModel, Field
from langchain.output_parsers import PydanticOutputParser

class Song(BaseModel):
    name: str = Field(description="Name of the Song")
    genre: list[str] = Field(description="List of the Genre")  # typing hint for v2
    #movie: str = Field(description="Movie name of the Song")

parser = PydanticOutputParser(pydantic_object=Song)
print(parser.get_format_instructions())


The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"name": {"description": "Name of the Song", "title": "Name", "type": "string"}, "genre": {"description": "List of the Genre", "items": {"type": "string"}, "title": "Genre", "type": "array"}}, "required": ["name", "genre"]}
```


In [188]:
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate,AIMessagePromptTemplate 
system_template = SystemMessagePromptTemplate.from_template("You are a helpful AI Song Recommendation Engine")
human_template = HumanMessagePromptTemplate.from_template("what is the most famous song by {singer_name}. \n{output_format_instructions}")

In [174]:
chat_template = ChatPromptTemplate.from_messages([system_template, human_template])

In [176]:
chain = chat_template | chat_model | parser
input = {"singer_name":"sonu nigam", "output_format_instructions": parser.get_format_instructions()}
chain.invoke(input)

Song(name='Kal Ho Naa Ho', genre=['Bollywood', 'Romantic', 'Soundtrack'])

In [184]:
input = {"singer_name":"arjit singh", "output_format_instructions": parser.get_format_instructions()}
chain.invoke(input)

Song(name='Tum Hi Ho', genre=['Bollywood', 'Romantic', 'Pop'])